# Moai - Kaggle/Colab Self-Contained Pipeline Notebook
 This notebook provides a self-contained, step-by-step pipeline suitable for
 Kaggle/Colab. It bundles the pipeline stages into notebook cells.


In [ ]:
# 1) Install Dependencies


import os, sys

try:
    import pandas as pd
    import numpy as np
    import sklearn
    import sympy
    import joblib
except Exception:
    print('Some packages missing; installing minimal dependencies...')
    # Use the notebook pip magic where available
    if 'IPython' in sys.modules:
        get_ipython().system('pip install -q pandas numpy scikit-learn sympy joblib')
    else:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'numpy', 'scikit-learn', 'sympy', 'joblib'])
try:
    import transformers
    import torch
    print('Transformers and torch are installed')
except Exception:
    print('Transformers or torch not installed; you can `pip install transformers torch` if you plan to run a real LLM')


In [ ]:
# Optional: Heavy LLM packages & model environment configuration
# Use this cell if you plan to run local HF models (quantized with bitsandbytes or full precision)
try:
    import transformers, accelerate, bitsandbytes, sentencepiece
    print('Heavy LLM packages are already installed')
except Exception:
    print('Installing heavy LLM packages: transformers, accelerate, bitsandbytes, sentencepiece')
    try:
        get_ipython().system('pip install -q transformers accelerate bitsandbytes sentencepiece')
    except Exception:
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers', 'accelerate', 'bitsandbytes', 'sentencepiece'])

# Configure model & quantization: adjust to your chosen HF model and quantization for Kaggle/Colab runtime
import os
# --- Kaggle/Colab Best Practice ---
# To use large models like Qwen2-72B without internet access in a Kaggle notebook,
# 1. Search for the model on Kaggle Datasets (e.g., "Qwen2 72B Instruct GGUF").
# 2. Add the dataset to your notebook. This will make the model files available under /kaggle/input/.
# 3. Set the AIMO_MODEL environment variable to the local path of the model directory.
# Example: os.environ['AIMO_MODEL'] = '/kaggle/input/qwen2-72b-instruct-gguf/Qwen2-72B-Instruct-GGUF'

# Note: Using a 72B model on free-tier Kaggle/Colab may cause memory issues.
# It is recommended to use a smaller model (like 1.5B or 7B) or a quantized version (GGUF/AWQ).

# Force set the environment variables to ensure the correct model is used
# Resolve common HF cache shortcut names (e.g., "models--Owner--Repo") to the
# local HF hub cache path when present. This helps when the model was downloaded
# into the Hugging Face cache (e.g., under ~/.cache/huggingface/hub/).
raw_model_env = os.getenv('AIMO_MODEL', 'models--Qwen--Qwen2.5-Math-72B-Instruct')
resolved_model = raw_model_env
try:
    if isinstance(raw_model_env, str) and raw_model_env.startswith('models--'):
        hf_cache_root = os.path.join(os.path.expanduser('~'), '.cache', 'huggingface', 'hub')
        candidate = os.path.join(hf_cache_root, raw_model_env)
        if os.path.isdir(candidate):
            resolved_model = candidate
        else:
            # also try alternative cache location
            alt_cache = os.path.join(os.path.expanduser('~'), '.cache', 'huggingface', 'hub', raw_model_env)
            if os.path.isdir(alt_cache):
                resolved_model = alt_cache
except Exception:
    # fallback to whatever was provided
    resolved_model = raw_model_env
os.environ['AIMO_MODEL'] = resolved_model
os.environ['AIMO_QUANTIZATION'] = os.getenv('AIMO_QUANTIZATION', '4bit') # Use 4bit for large models
print('Resolved AIMO_MODEL:', os.environ.get('AIMO_MODEL'))
print('AIMO_QUANTIZATION:', os.environ.get('AIMO_QUANTIZATION'))


In [ ]:
# 4) Helper functions
from pathlib import Path
import json, subprocess, shlex

def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with open(path,'r',encoding='utf8') as f:
        return [json.loads(line) for line in f if line.strip()]

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path,'w',encoding='utf8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def extract_answer_from_text(text):
    # Try to parse common answer patterns like Boxed or final: 123
    if text is None:
        return None
    import re
    m = re.search(r"\\boxed\{([^}]+)\}", text)
    if m:
        return m.group(1).strip()
    # common final answer markers
    m = re.search(r"Final answer[:\s]*([0-9a-zA-Z.+-*/ ]+)", text, re.I)
    if m:
        return m.group(1).strip()
    # fallback: last number-ish token
    m = re.findall(r"[-+]?[0-9]*\.?[0-9]+(?:[eE][-+]?[0-9]+)?", text)
    if m:
        return m[-1]
    return None

def safe_run_code(code, timeout=5):
    # Use a subprocess to run Python and capture stdout; this prevents accidental shared state in the notebook
    cmd = [sys.executable, '-c', code]
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, check=False)
        return {'stdout': proc.stdout, 'stderr': proc.stderr, 'returncode': proc.returncode}
    except subprocess.TimeoutExpired:
        return {'stdout': '', 'stderr': 'timeout', 'returncode': -1}

def log_result(item, prefix=''): 
    print(prefix + json.dumps(item, ensure_ascii=False))

In [ ]:
# 3) Environment & Reproducibility

try:
    import torch
    print('Torch version:', torch.__version__ if hasattr(torch, '__version__') else 'unknown')
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print('CUDA device available: ', torch.cuda.get_device_name(0))
    else:
        device = torch.device('cpu')
        print('CUDA not available; using CPU')
    print('Device:', device)
except Exception:
    device = 'cpu'
    print('torch not available; using CPU')

# Check bitsandbytes & accelerate availability
try:
    import bitsandbytes as bnb
    print('bitsandbytes version:', bnb.__version__ if hasattr(bnb, '__version__') else 'unknown')
except Exception:
    print('bitsandbytes not installed or not available')
try:
    import accelerate as acc
    print('accelerate version:', acc.__version__ if hasattr(acc, '__version__') else 'unknown')
except Exception:
    print('accelerate not installed or not available')


In [ ]:
# 4) Helper functions
from pathlib import Path
import json, subprocess, shlex

def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with open(path,'r',encoding='utf8') as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path,'w',encoding='utf8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def extract_answer_from_text(text):
    # Try to parse common answer patterns like Boxed or final: 123
    if text is None:
        return None
    import re
    m = re.search(r"\\boxed\{([^}]+)\}", text)
    if m:
        return m.group(1).strip()
    # common final answer markers
    m = re.search(r"Final answer[:\s]*([0-9a-zA-Z.+-*/ ]+)", text, re.I)
    if m:
        return m.group(1).strip()
    # fallback: last number-ish token
    m = re.findall(r"[-+]?[0-9]*\.?[0-9]+(?:[eE][-+]?[0-9]+)?", text)
    if m:
        return m[-1]
    return None


def safe_run_code(code, timeout=5):
    # Use a subprocess to run Python and capture stdout; this prevents accidental shared state in the notebook
    cmd = [sys.executable, '-c', code]
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, check=False)
        return {'stdout': proc.stdout, 'stderr': proc.stderr, 'returncode': proc.returncode}
    except subprocess.TimeoutExpired:
        return {'stdout': '', 'stderr': 'timeout', 'returncode': -1}


def log_result(item, prefix=''): 
    print(prefix + json.dumps(item, ensure_ascii=False))


In [ ]:
# 4b) Embedded pipeline core modules (Router, Executor, Solver, Verifier, Decomposer, Orchestrator)
# These definitions are derived from the `AIMO_core/moai_orchestrator.py` self-contained runner
import os, json, re, uuid
from datetime import datetime

class _Config:
    LOG_PATH = "AIMO_core/logs/eval_log.jsonl"
    DECOMPOSITION_COMPLEXITY_THRESHOLD = 15
    USE_VOTING = False
    NUM_CANDIDATES = 3
    CANDIDATE_TEMPS = [0.1, 0.3]

config = _Config()

# lightweight reasoning utils

def extract_answer(reasoning_text: str):
    if not reasoning_text:
        return None
    m = re.search(r"\\boxed\{([^}]*)\}", reasoning_text)
    if m:
        return m.group(1).strip()
    m = re.search(r"(-?\\d+(?:\\.\\d+)?)", reasoning_text)
    return m.group(1) if m else None

def log_result(obj: dict, path=None):
    path = path or config.LOG_PATH
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(obj, ensure_ascii=False, default=str) + "\n")

def make_ids(problem_text: str) -> dict:
    return {
        'run_id': str(uuid.uuid4()),
        'problem_hash': hash(problem_text),
        'timestamp': datetime.utcnow().isoformat()
    }


In [ ]:
# 7) Dataset load and sample creation
from pathlib import Path

DATA_IN = Path('AIMO_core/data/aimo_problems.jsonl')
DATA_OUT = Path('AIMO_core/data/numina_training_5k.jsonl')

if DATA_IN.exists():
    print('Loading dataset from', DATA_IN)
    problems = [r['problem'] if isinstance(r, dict) and 'problem' in r else str(r) for r in read_jsonl(DATA_IN)]
    print('Loaded', len(problems), 'problems')
else:
    print(DATA_IN, 'not found. Creating a tiny synthetic dataset for demo')
    problems = [
        'What is 12 + 30?',
        'Compute 17 * 5 - 3',
        'A rectangle has width 5 and height 7. What is its area?'
    ]

# limit for the demo
print('First 3 problems:\n', problems[:3])


In [ ]:
# 8) Run fully-featured pipeline
N = 3 # Use 3 for quick test
results = []

# Initialize the fully-featured orchestrator
# This will automatically use the model specified in the 'AIMO_MODEL' environment variable.
embedded_orch = EmbeddedPipelineOrchestrator()

for i, prob in enumerate(problems[:N]):
    print(f'\n==== Problem {i+1} ====')
    res = embedded_orch.solve_problem(domain='math', variables={}, problem_text=prob)
    print(f'Problem: {prob}')
    print(f'Result: {res}')
    results.append({'id': i, 'problem': prob, 'res': res})

# Save results
write_jsonl(DATA_OUT, results)
print(f'\nSaved {len(results)} rows to {DATA_OUT}')


In [ ]:
# 9) Basic sanity-check / evaluation
out_rows = read_jsonl(DATA_OUT)
print('Loaded output rows:', len(out_rows))
if out_rows:
    print(json.dumps(out_rows[0], ensure_ascii=False, indent=2))

# A very small 'verifier' example: check that answers are numbers if expected
import re
num_count = sum(1 for r in out_rows if r['res'].get('answer') and re.search(r"[-+]?[0-9]*\.?[0-9]+", str(r['res']['answer'])))
print(f"{num_count}/{len(out_rows)} answers look numeric")

In [ ]:
# Quick: validate environment for heavy model loads (optional)
try:
    get_ipython().system('python AIMO_core/check_model_env.py')
except Exception as e:
    print('Could not run check script in this environment:', e)

In [ ]:
# Example usage with cache dir set (recommended on Kaggle):
# Quick demo: set HF cache dir to a workspace folder where you pre-download the model
import os
hf_cache = '/kaggle/working/hf_cache'  # change for your environment
os.environ['HF_HOME'] = hf_cache
os.environ['TRANSFORMERS_CACHE'] = hf_cache
print('HF_HOME:', os.environ['HF_HOME'])
print('TRANSFORMERS_CACHE:', os.environ['TRANSFORMERS_CACHE'])
# Run CLI (example):
# !python AIMO_core/kaggle_math_pipeline.py --model qwen-1.5b --quant 8bit --cache-dir /kaggle/working/hf_cache

In [ ]:
# 10) Kaggle / Colab specific utilities
# Colab: mount drive
try:
    from google.colab import drive
    DRIVE_COLAB = True
    print('We are in Colab; mounting drive is optional')
except Exception:
    DRIVE_COLAB = False

# Kaggle uses /kaggle/working for output files; copying is optional
print('Colab drive mounted:', DRIVE_COLAB)

# Convert notebook to script (optional) - useful for submission as a single file
try:
    get_ipython().system('jupyter nbconvert --to script "AIMO_kaggle_pipeline.ipynb" --output moai_orchestrator.py')
except Exception as e:
    print('Failed to convert to script (safe to ignore if nbconvert not installed):', e)

# If running on Kaggle, copy results to /kaggle/working
if is_kaggle:
    try:
        import shutil
        out_script = Path('moai_orchestrator.py')
        # if script exists, move/copy or do nothing
        if out_script.exists():
            shutil.copy2(out_script, Path('/kaggle/working') / out_script.name)
        # copy saved outputs
        if DATA_OUT.exists():
            shutil.copy2(DATA_OUT, Path('/kaggle/working') / DATA_OUT.name)
        print('Copied results to /kaggle/working')
    except Exception as e:
        print('Could not copy to /kaggle/working:', e)


# Notes & Tips

- This notebook is a self-contained, demonstration-ready pipeline. It is intentionally lightweight and uses a fallback LLM stub if Hugging Face transformers is not available.

In [ ]:
# Embedded pipeline modules (fully-featured)
import os, json, re, uuid, math
from datetime import datetime
from collections import defaultdict

# --- Configuration ---
class _Config:
    LOG_PATH = "AIMO_core/logs/eval_log.jsonl"
    DECOMPOSITION_COMPLEXITY_THRESHOLD = 15

config = _Config()

# --- Helper Functions ---
def make_ids(problem_text: str) -> dict:
    return {
        'run_id': str(uuid.uuid4()),
        'problem_hash': hash(problem_text),
        'timestamp': datetime.utcnow().isoformat()
    }

def log_result(obj: dict):
    path = config.LOG_PATH
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(obj, ensure_ascii=False, default=str) + "\n")

# --- LLM Client ---
class LocalLLM:
    """A light-weight LLM stub that uses a Hugging Face transformer pipeline."""
    
Initializing LocalLLM with model: {model_name}